06 streamlit

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")


def find_project_root() -> Path:
    """Find the project root whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")


ROOT = find_project_root()
DATASETS = ROOT / "datasets"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: C:\Users\HP\Desktop\Customer 360 Intelligence


## 1. Validate required files and model artifacts

In [2]:
required_files = {
    "customer_360_features.csv": [
        "customer_id", "state", "city", "total_orders", "total_spend", "avg_order_value",
        "recency_days", "frequency", "monetary", "favorite_category", "rfm_segment",
        "cluster_segment", "predicted_clv", "churn_probability", "churn_risk_band", "clv_band",
    ],
    "segment_summary.csv": ["rfm_segment", "customers", "revenue", "avg_clv", "avg_churn_probability"],
    "product_sentiment.csv": ["category_name", "avg_rating", "review_count", "sentiment_label"],
    "recommendations.csv": ["customer_id", "rank", "recommended_category", "method", "reason"],
    "model_feature_importance.csv": ["feature", "churn_importance", "clv_importance"],
}
required_models = ["churn_model.pkl", "clv_model.pkl", "segment_model.pkl", "sentiment_model.pkl"]

status_rows = []
for file_name, expected_columns in required_files.items():
    path = PROCESSED / file_name
    assert path.exists(), f"Missing output: {path}. Run Notebook 05."
    frame = pd.read_csv(path)
    missing_columns = sorted(set(expected_columns) - set(frame.columns))
    assert not missing_columns, f"{file_name} is missing columns: {missing_columns}"
    assert len(frame) > 0, f"{file_name} is empty."
    status_rows.append({"artifact": file_name, "type": "data", "rows": len(frame), "status": "ready"})

for model_name in required_models:
    path = MODELS / model_name
    assert path.exists(), f"Missing model: {path}. Run Notebook 05."
    status_rows.append({"artifact": model_name, "type": "model", "rows": None, "status": "ready"})

release_status = pd.DataFrame(status_rows)
display(release_status)

,artifact,type,rows,status
0,customer_360_features.csv,data,94983.0,ready
1,segment_summary.csv,data,6.0,ready
2,product_sentiment.csv,data,22.0,ready
3,recommendations.csv,data,474915.0,ready
4,model_feature_importance.csv,data,6.0,ready
5,churn_model.pkl,model,NaN,ready
6,clv_model.pkl,model,NaN,ready
7,segment_model.pkl,model,NaN,ready
8,sentiment_model.pkl,model,NaN,ready


## 2. Validate customer score ranges and uniqueness

In [3]:
customer = pd.read_csv(PROCESSED / "customer_360_features.csv")
assert customer["customer_id"].is_unique, "Customer output must contain one row per canonical customer."
assert customer["churn_probability"].between(0, 1).all(), "Churn probabilities must be between 0 and 1."
assert customer["predicted_clv"].ge(0).all(), "Predicted CLV cannot be negative."
assert customer["total_spend"].ge(0).all(), "Total spend cannot be negative."
assert customer["rfm_segment"].notna().all(), "Every customer needs an RFM segment."

quality_summary = customer[[
    "total_spend", "total_orders", "recency_days", "predicted_clv", "churn_probability",
]].describe().T
display(quality_summary)

,count,mean,std,min,25%,50%,75%,max
total_spend,94983.0,142.071747,216.074999,0.850000,47.900000,89.890000,155.000000,13440.000000
total_orders,94983.0,1.033859,0.210811,1.000000,1.000000,1.000000,1.000000,16.000000
recency_days,94983.0,243.334207,152.984597,0.000000,119.000000,224.000000,352.000000,729.000000
predicted_clv,94983.0,178.960732,244.212469,0.000000,51.240383,103.612160,203.352500,2073.523200
churn_probability,94983.0,0.402418,0.141841,0.040414,0.281558,0.397905,0.518931,0.724966


## 3. Launch command

In [4]:
print("Release checks passed.")
print("From the project root, run:")
print("streamlit run streamlit_app/app.py")

Release checks passed.
From the project root, run:
streamlit run streamlit_app/app.py
